# 03 — Pick Best Per Class

Each 02 notebook saved its own best model's OOF and test predictions
(`{class}_oof.parquet`, `{class}_test.parquet`) plus the tuning metric
(`{class}_score.json`). This notebook picks the best per class and aligns
them into the OOF/test prediction matrices that 04 and 05 consume.
No sklearn artifact path is assumed — the NN participates through its
saved predictions.


In [ ]:
from src.utils import load_env

load_env()

from src.constants import DATA_PROCESSED

input_dir = str(DATA_PROCESSED)

output_dir = str(DATA_PROCESSED)

cv_folds = 5

random_state = 42

experiments = {
    "linear": "linear_models",
    "gbdt": "gbdt_models",
    "nn": "nn_models",
}

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

In [ ]:
# ── Load labels (predictions come from the per-class 02 artifacts) ──
y_train = pd.read_parquet(f"{input_dir}/y_train.parquet")["y"].reset_index(drop=True)
y_test = pd.read_parquet(f"{input_dir}/y_test.parquet")["y"].reset_index(drop=True)
print(f"Train: {len(y_train)}, Test: {len(y_test)}")

In [ ]:
# Each 02 notebook saved its own best model's OOF and test predictions
# ({name}_oof.parquet / {name}_test.parquet) plus the tuning metric
# ({name}_score.json). 03 picks the best per class and aligns them into
# the matrices 04/05 consume. All three classes (linear, gbdt, nn) flow
# through this path — no `runs:/.../model` sklearn artifact is assumed.
names = list(experiments.keys())
oof_preds = np.zeros((len(y_train), len(names)))
test_preds = np.zeros((len(y_test), len(names)))

for i, name in enumerate(names):
    oof = pd.read_parquet(f"{input_dir}/{name}_oof.parquet")["pred"].to_numpy()
    test = pd.read_parquet(f"{input_dir}/{name}_test.parquet")["pred"].to_numpy()
    if len(oof) != len(y_train) or len(test) != len(y_test):
        raise ValueError(
            f"{name} predictions ({len(oof)}, {len(test)}) don't match labels "
            f"({len(y_train)}, {len(y_test)})"
        )
    with open(f"{input_dir}/{name}_score.json") as f:
        score = json.load(f)
    oof_preds[:, i] = oof
    test_preds[:, i] = test
    print(f"Best {name:6s}: {score['metric']} = {score['score']:.4f}")

pd.DataFrame(oof_preds, columns=names).to_parquet(f"{output_dir}/oof_preds.parquet")
pd.DataFrame(test_preds, columns=names).to_parquet(f"{output_dir}/test_preds.parquet")
pd.Series(y_train, name="match_won").to_frame().to_parquet(f"{output_dir}/y_train_full.parquet")
print(f"OOF predictions saved: {oof_preds.shape}, test predictions: {test_preds.shape}")

In [ ]:
# ── Consolidate pinned base-model identities from 02 ──
# Each 02 notebook wrote {name}_model_version.json with its registered
# base model's exact version, run ID, and model URI. 04 logs these pins
# and the build step enforces them, so no alias or `latest` resolution
# happens anywhere downstream.
base_pins = {}
for name in names:
    with open(f"{input_dir}/{name}_model_version.json") as f:
        base_pins[name] = json.load(f)
with open(f"{output_dir}/base_pins.json", "w") as f:
    json.dump(base_pins, f, indent=2)
for name, pin in base_pins.items():
    print(f"  {name}: {pin['registered_model_name']} v{pin['version']} (run {pin['run_id'][:8]})")

In [ ]:
# ── Compare best-per-class ROC-AUC (OOF, comparable across classes) ──
for i, name in enumerate(names):
    score = roc_auc_score(y_train, oof_preds[:, i])
    print(f"  {name:8s} OOF ROC-AUC: {score:.4f}")